In [0]:
dbutils.widgets.text("Environment", "dev", "Set the current environment/catalog name")
env = dbutils.widgets.get("Environment")

In [0]:
%run ./02-setup

In [0]:
# Parameterise these before running
catalog = env
db_name = f"sbit_{env}"

In [0]:
#cleanup from setup file (drop db is exists)
cleanup(catalog, db_name)

In [0]:
#running the run-script files in testing 
try:
    #calling 07-run script, run for 10 mins, passing env (dev) and runtype = once (batch mode)
    result = dbutils.notebook.run("./07-run", 600, {"Environment": env, "RunType": "once"}) 
    print("Notebook run succeeded:", result)
except Exception as e:
    print("Notebook run failed. Reason:")
    print(e)

#.run() parameters: script to run, timeout in seconds, parameters to pass to the script

In [0]:
%run ./03-history-loader

In [0]:
validate_setup(catalog, db_name) # from setup file
validate(catalog, db_name)       # from history-loader file

#checking they are created
#checking date_lookup has data (only one that has any data yet)

In [0]:
#want to simulate incoming data so import 10-producer below

In [0]:
%run ./10-producer

In [0]:
#call funcs from 10-producer to import data (set 1 only)
produce_data(1) # set 1
validate_data(1)

In [0]:
%run ./04-bronze

In [0]:
%run ./05-silver

In [0]:
%run ./06-gold

In [0]:
validate_bronze(catalog, db_name, 1)
validate_silver(catalog, db_name, 1)
validate_gold(catalog, db_name, test_data_dir, 1)

In [0]:
# bring in the set 2 data to raw folder 
produce_data(2)
validate_data(2)

In [0]:
#run again to load the data into the tables
#due to the checkpoint, it will only find the new files (set 2 files)

try:
    result = dbutils.notebook.run("./07-run", 600, {"Environment": env, "RunType": "once"})
    print("Notebook run succeeded:", result)
except Exception as e:
    print("Notebook run failed. Reason:")
    print(e)

In [0]:
validate_bronze(catalog, db_name, 2)
validate_silver(catalog, db_name, 2)
validate_gold(catalog, db_name, test_data_dir, 2)